# SPOTnet — Step 02: Baseline Training (BCE + Dice)

This notebook trains the **SPOTnet Baseline** — a lightweight UNet trained on the specular patch library using BCE + Dice loss at a single scale (128×128).

**Pipeline:**
```
Patch Library → Single-Scale Dataset → UNet → BCE + Dice Loss → Baseline Model
```

**Output:** `runs/detector_baseline/best.pt`

## 1. Imports & Config

In [ ]:
import os, glob, random, shutil
import numpy as np
import cv2
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 1337
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# Paths
PATCH_DIR = 'real_specular_library_v1/patches'
MASK_DIR  = 'real_specular_library_v1/masks'
RUN_DIR   = 'runs/detector_baseline'
os.makedirs(RUN_DIR, exist_ok=True)

# Hyperparameters
BASE_CH  = 32
EPOCHS   = 20
BATCH    = 32
LR       = 3e-4
POS_W    = 5.0
IMG_SIZE = 128

## 2. Model Architecture — UNet

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, 1, 1, bias=False), nn.BatchNorm2d(cout), nn.SiLU(inplace=True),
            nn.Conv2d(cout, cout, 3, 1, 1, bias=False), nn.BatchNorm2d(cout), nn.SiLU(inplace=True))
    def forward(self, x): return self.net(x)

class UNet(nn.Module):
    def __init__(self, base=32):
        super().__init__()
        self.e1 = ConvBlock(3, base)
        self.e2 = ConvBlock(base, base*2)
        self.e3 = ConvBlock(base*2, base*4)
        self.b  = ConvBlock(base*4, base*4)
        self.d2 = ConvBlock(base*4+base*2, base*2)
        self.d1 = ConvBlock(base*2+base, base)
        self.head = nn.Conv2d(base, 1, 1)

    def forward(self, x):
        s1 = self.e1(x)
        s2 = self.e2(F.avg_pool2d(s1, 2))
        s3 = self.e3(F.avg_pool2d(s2, 2))
        b  = self.b(s3)
        x  = F.interpolate(b,  scale_factor=2, mode='bilinear', align_corners=False)
        x  = self.d2(torch.cat([x, s2], dim=1))
        x  = F.interpolate(x,  scale_factor=2, mode='bilinear', align_corners=False)
        x  = self.d1(torch.cat([x, s1], dim=1))
        return self.head(x)

print(f'Parameters: {sum(p.numel() for p in UNet().parameters()):,}')

## 3. Dataset

In [ ]:
def to_tensor(img):
    if not img.flags['C_CONTIGUOUS']: img = np.ascontiguousarray(img)
    return torch.from_numpy(img).float().permute(2,0,1) / 255.0

class PatchDS(Dataset):
    def __init__(self, patch_dir, mask_dir, size=128):
        self.patches = sorted(glob.glob(os.path.join(patch_dir, '*.png')))
        mmap = {os.path.basename(m): m for m in glob.glob(os.path.join(mask_dir, '*.png'))}
        self.pairs = [(p, mmap[os.path.basename(p)]) for p in self.patches if os.path.basename(p) in mmap]
        self.size = size
        print(f'Pairs: {len(self.pairs)}')

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        p, m = self.pairs[idx]
        img  = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
        mask = cv2.imread(m, 0)
        img  = cv2.resize(img,  (self.size, self.size), interpolation=cv2.INTER_AREA)
        mask = cv2.resize(mask, (self.size, self.size), interpolation=cv2.INTER_NEAREST)
        mask = (mask > 127).astype(np.uint8)
        # Augmentation
        if random.random() < 0.5: img = np.fliplr(img);  mask = np.fliplr(mask)
        if random.random() < 0.5: img = np.flipud(img);  mask = np.flipud(mask)
        if random.random() < 0.5:
            k = random.choice([1,2,3])
            img = np.rot90(img, k); mask = np.rot90(mask, k)
        X = to_tensor(np.ascontiguousarray(img))
        M = torch.from_numpy(np.ascontiguousarray(mask.astype(np.float32)))[None]
        return X, M

ds = PatchDS(PATCH_DIR, MASK_DIR, size=IMG_SIZE)
dl = DataLoader(ds, batch_size=BATCH, shuffle=True, num_workers=4, drop_last=True)

## 4. Loss Functions

In [ ]:
def dice_loss(logits, targets, eps=1e-6):
    p   = torch.sigmoid(logits)
    num = 2*(p*targets).sum(dim=(2,3))
    den = (p+targets).sum(dim=(2,3)) + eps
    return (1 - num/den).mean()

def criterion(logits, targets):
    bce  = F.binary_cross_entropy_with_logits(logits, targets,
               pos_weight=torch.tensor([POS_W], device=logits.device))
    dice = dice_loss(logits, targets)
    return bce + dice

## 5. Training

In [ ]:
net   = UNet(BASE_CH).to(DEVICE)
opt   = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type=='cuda'))

best_loss = float('inf')
for ep in range(1, EPOCHS+1):
    net.train()
    losses = []
    for X, M in tqdm(dl, desc=f'Ep {ep}/{EPOCHS}'):
        X, M = X.to(DEVICE), M.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=(DEVICE.type=='cuda')):
            loss = criterion(net(X), M)
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update()
        losses.append(loss.item())
    sched.step()
    avg = np.mean(losses)
    print(f'Ep {ep:03d} | loss: {avg:.4f}')
    if avg < best_loss:
        best_loss = avg
        torch.save({'net': net.state_dict()}, f'{RUN_DIR}/best.pt')
    torch.save({'net': net.state_dict()}, f'{RUN_DIR}/last.pt')

print(f'\nBest loss: {best_loss:.4f}')
print(f'Saved: {RUN_DIR}/best.pt')

## 6. Quick Visual Check

In [ ]:
import math, matplotlib.pyplot as plt

@torch.no_grad()
def tiled_inference(net, img_rgb, tile=384, overlap=96):
    net.eval()
    H, W = img_rgb.shape[:2]
    X = to_tensor(img_rgb).unsqueeze(0).to(DEVICE)
    stride = tile - overlap
    pad_h = (math.ceil((H-tile)/stride)*stride+tile-H) if H>tile else tile-H
    pad_w = (math.ceil((W-tile)/stride)*stride+tile-W) if W>tile else tile-W
    X = F.pad(X, (0,pad_w,0,pad_h), mode='reflect')
    Hp, Wp = X.shape[-2:]
    acc = torch.zeros((H,W), device=DEVICE)
    cnt = torch.zeros((H,W), device=DEVICE)
    for y in range(0, Hp-tile+1, stride):
        for x in range(0, Wp-tile+1, stride):
            pred = torch.sigmoid(net(X[:,:,y:y+tile,x:x+tile]))[0,0]
            ye, xe = min(y+tile,H), min(x+tile,W)
            acc[y:ye,x:xe] += pred[:ye-y,:xe-x]
            cnt[y:ye,x:xe] += 1
    return (acc/cnt.clamp(min=1)).cpu().numpy()

# Load best model
ck = torch.load(f'{RUN_DIR}/best.pt', map_location='cpu')
net.load_state_dict(ck['net'])

# Test on 3 frames
frames = sorted(glob.glob('frames/*.jpg') + glob.glob('frames/*.png'))[:3]
fig, axes = plt.subplots(3, 3, figsize=(12, 9))
for i, fp in enumerate(frames):
    img = cv2.cvtColor(cv2.imread(fp), cv2.COLOR_BGR2RGB)
    soft = tiled_inference(net, img)
    pred = (soft > 0.5).astype(np.uint8) * 255
    overlay = img.copy(); overlay[pred > 0] = [255, 0, 0]
    axes[i,0].imshow(img);            axes[i,0].set_title('Original'  if i==0 else '')
    axes[i,1].imshow(pred, cmap='gray'); axes[i,1].set_title('Mask'   if i==0 else '')
    axes[i,2].imshow(overlay);        axes[i,2].set_title('Overlay'   if i==0 else '')
    for ax in axes[i]: ax.axis('off')
plt.tight_layout(); plt.show()